In [38]:
# Cell 1: Setup and imports
import pandas as pd
import boto3
from io import BytesIO
import sys
from pathlib import Path

# Add src to path
root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(root / 'src'))
from player_name_utils import normalize_player_name

# Config
SEASON = '2025-26'
RIM_SCORER_PCT = 40
DATE = '2026-01-18'

In [39]:
# Cell 2: Load scorer type data
def load_scorer_types(season=SEASON, rim_scorer_pct=RIM_SCORER_PCT):
    """Load player scorer type classifications from S3"""
    print(f"📊 Loading player scorer type data...")
    
    s3_key = f"data/03_intermediate/player_props_with_actuals_{season}_rim{rim_scorer_pct}.csv"
    
    s3 = boto3.client('s3')
    bucket = 'nba-betting-mt'
    obj = s3.get_object(Bucket=bucket, Key=s3_key)
    
    df = pd.read_csv(BytesIO(obj['Body'].read()))
    
    if 'scorer_type' not in df.columns:
        print(f"   ⚠️  WARNING: scorer_type column not found")
        return {}, df
    
    # Normalize player names
    df['PLAYER_NAME_NORMALIZED'] = df['PLAYER_NAME'].apply(normalize_player_name)
    
    # Create mapping
    scorer_map = df[['PLAYER_NAME_NORMALIZED', 'scorer_type']].dropna().drop_duplicates('PLAYER_NAME_NORMALIZED').set_index('PLAYER_NAME_NORMALIZED')['scorer_type'].to_dict()
    
    rim_count = sum(1 for v in scorer_map.values() if 'Rim' in str(v))
    perim_count = sum(1 for v in scorer_map.values() if 'Perimeter' in str(v))
    
    print(f"   ✅ Loaded scorer types for {len(scorer_map)} players")
    print(f"      Rim Attackers (≥{rim_scorer_pct}%): {rim_count}")
    print(f"      Perimeter (<{rim_scorer_pct}%): {perim_count}")
    
    return scorer_map, df

scorer_map, df_full = load_scorer_types()

📊 Loading player scorer type data...
   ✅ Loaded scorer types for 488 players
      Rim Attackers (≥40%): 185
      Perimeter (<40%): 303


In [40]:
# Cell 3: Load today's props
def load_todays_props(date=DATE):
    """Load today's player props from S3"""
    print(f"\n📊 Loading today's player props ({date})...")
    
    s3_key = f"data/04_output/plays/role_spread_points_model/2d/{date}_top3.csv"
    
    s3 = boto3.client('s3')
    bucket = 'nba-betting-mt'
    
    try:
        obj = s3.get_object(Bucket=bucket, Key=s3_key)
        df = pd.read_csv(BytesIO(obj['Body'].read()))
        print(f"   ✅ Loaded {len(df)} player props")
        return df
    except Exception as e:
        print(f"   ❌ Error: {e}")
        return pd.DataFrame()

df_props = load_todays_props()
df_props.head()


📊 Loading today's player props (2026-01-18)...
   ✅ Loaded 8 player props


,date,season,player,team,opponent,bet_side,line,spread,line_tier,spread_bin,...,expected_roi,edge_vs_baseline,hit_rate,games_in_sample,game_time,bookmakers,num_bookmakers,bookmaker_details_over,bookmaker_details_under,edge_vs_breakeven
0,2026-01-18,2025-26,Bruce Brown,DEN,CHA,UNDER,6.5,1.5,5-10 (Bench),Pick'em (-2 to +2),...,13.6,7.9,59.5,237,2026-01-18 20:10:00-05:00,"BetOnline.ag, BetRivers, Bovada, Caesars, Draf...",6,"[{""bookmaker"": ""BetOnline.ag"", ""line"": 6.5, ""o...","[{""bookmaker"": ""BetOnline.ag"", ""line"": 6.5, ""o...",7.12
1,2026-01-18,2025-26,Jalen Pickett,DEN,CHA,UNDER,8.5,1.5,5-10 (Bench),Pick'em (-2 to +2),...,13.6,7.9,59.5,237,2026-01-18 20:10:00-05:00,"BetMGM, BetOnline.ag, Bovada, Caesars, DraftKi...",6,"[{""bookmaker"": ""BetMGM"", ""line"": 8.5, ""odds"": ...","[{""bookmaker"": ""BetMGM"", ""line"": 8.5, ""odds"": ...",7.12
2,2026-01-18,2025-26,Moussa Diabate,CHA,DEN,UNDER,7.5,-1.5,5-10 (Bench),Pick'em (-2 to +2),...,13.6,7.9,59.5,237,2026-01-18 20:10:00-05:00,"BetRivers, Bovada, Caesars, DraftKings, FanDuel",5,"[{""bookmaker"": ""BetRivers"", ""line"": 7.5, ""odds...","[{""bookmaker"": ""BetRivers"", ""line"": 7.5, ""odds...",7.12
3,2026-01-18,2025-26,Rui Hachimura,LAL,TOR,UNDER,7.5,-1.0,5-10 (Bench),Pick'em (-2 to +2),...,13.6,7.9,59.5,237,2026-01-18 21:40:00-05:00,"BetMGM, BetOnline.ag, Caesars, DraftKings",4,"[{""bookmaker"": ""BetMGM"", ""line"": 7.5, ""odds"": ...","[{""bookmaker"": ""BetMGM"", ""line"": 7.5, ""odds"": ...",7.12
4,2026-01-18,2025-26,Sandro Mamukelashvili,TOR,LAL,UNDER,9.5,1.0,5-10 (Bench),Pick'em (-2 to +2),...,13.6,7.9,59.5,237,2026-01-18 21:40:00-05:00,"BetMGM, BetOnline.ag, BetRivers, Bovada, Caesa...",8,"[{""bookmaker"": ""BetMGM"", ""line"": 9.5, ""odds"": ...","[{""bookmaker"": ""BetMGM"", ""line"": 9.5, ""odds"": ...",7.12


In [41]:
# Cell 4: Find unclassified players
players_today = df_props['player'].unique()
players_today_normalized = [normalize_player_name(p) for p in players_today]

unclassified = [p for i, p in enumerate(players_today) if players_today_normalized[i] not in scorer_map]

print(f"📊 Today's unique players: {len(players_today)}")
print(f"\n{'='*80}")
if unclassified:
    print(f"❌ Found {len(unclassified)} players WITHOUT scorer_type:\n")
    for player in unclassified:
        print(f"  • {player}")
else:
    print("✅ All players have scorer_type classification")
print(f"{'='*80}")

📊 Today's unique players: 8

✅ All players have scorer_type classification


In [42]:
# this is just for today^

In [43]:
...

Ellipsis

In [44]:
# looking for all players, now...

In [45]:
# we fixed it - before, because of the accent on his last name's E, Moussa Diabate did not have a scorer_type classification

In [46]:
# All Moussa rows
d=df_full[df_full['PLAYER_NAME'].str.contains('Moussa', case=False, na=False)]
print(d.shape)
d

(62, 50)


,PLAYER_ID,PLAYER_NAME,PLAYER_NAME_NORMALIZED,TEAM_ID,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,...,total_pts_season,pts_0_6_pct,scorer_type,rim_scorer_threshold,team_spread,team_spread_odds,team_moneyline,opponent_team,is_home,is_favorite
122,1631217,Moussa Diabaté,Moussa Diabate,1610612766,Charlotte Hornets,22500080,2025-10-22,CHA vs. BKN,W,21.233333,...,310.0,70.322581,Rim Attacker (≥40.0%),40.0,-5.318182,-109.818182,-205.727273,Brooklyn Nets,True,True
307,1630619,Moussa Cisse,Moussa Cisse,1610612742,Dallas Mavericks,22500004,2025-10-22,DAL vs. SAS,L,4.066667,...,85.0,61.176471,Rim Attacker (≥40.0%),40.0,-3.590909,-89.909091,-163.909091,San Antonio Spurs,True,True
616,1630619,Moussa Cisse,Moussa Cisse,1610612742,Dallas Mavericks,22500096,2025-10-24,DAL vs. WAS,L,0.276667,...,85.0,61.176471,Rim Attacker (≥40.0%),40.0,-9.409091,-108.272727,-456.727273,Washington Wizards,True,True
695,1631217,Moussa Diabaté,Moussa Diabate,1610612766,Charlotte Hornets,22500102,2025-10-25,CHA @ PHI,L,16.945000,...,310.0,70.322581,Rim Attacker (≥40.0%),40.0,4.500000,-111.909091,153.818182,Philadelphia 76ers,False,False
803,1631217,Moussa Diabaté,Moussa Diabate,1610612766,Charlotte Hornets,22500109,2025-10-26,CHA @ WAS,W,22.973333,...,310.0,70.322581,Rim Attacker (≥40.0%),40.0,1.500000,-109.181818,85.181818,Washington Wizards,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12711,1631217,Moussa Diabaté,Moussa Diabate,1610612766,Charlotte Hornets,22500563,2026-01-12,CHA @ LAC,L,35.113333,...,310.0,70.322581,Rim Attacker (≥40.0%),40.0,5.500000,-111.100000,168.454545,Los Angeles Clippers,False,False
12786,1630619,Moussa Cisse,Moussa Cisse,1610612742,Dallas Mavericks,22500561,2026-01-12,DAL vs. BKN,W,11.783333,...,85.0,61.176471,Rim Attacker (≥40.0%),40.0,-3.454545,-109.909091,-159.181818,Brooklyn Nets,True,True
13170,1630619,Moussa Cisse,Moussa Cisse,1610612742,Dallas Mavericks,22500583,2026-01-15,DAL vs. UTA,W,23.483333,...,85.0,61.176471,Rim Attacker (≥40.0%),40.0,-2.500000,-109.727273,-138.181818,Utah Jazz,True,True
13246,1631217,Moussa Diabaté,Moussa Diabate,1610612766,Charlotte Hornets,22500586,2026-01-15,CHA @ LAL,W,30.133333,...,310.0,70.322581,Rim Attacker (≥40.0%),40.0,3.350000,-110.300000,134.727273,Los Angeles Lakers,False,False


In [47]:
# see, moussa is a Rim Attacker


In [48]:
d2 = d.drop_duplicates(subset=['PLAYER_NAME', 'scorer_type'])

cols = ['PLAYER_NAME', 'scorer_type']

d2[cols]

,PLAYER_NAME,scorer_type
122,Moussa Diabaté,Rim Attacker (≥40.0%)
307,Moussa Cisse,Rim Attacker (≥40.0%)


In [49]:
# look for rows where he has no scorer_type
# Only Moussa rows WITHOUT scorer_type
reqs = (df_full['PLAYER_NAME'].str.contains('Moussa', case=False, na=False)) & (df_full['scorer_type'].isna())
df_full[reqs]

,PLAYER_ID,PLAYER_NAME,PLAYER_NAME_NORMALIZED,TEAM_ID,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,...,total_pts_season,pts_0_6_pct,scorer_type,rim_scorer_threshold,team_spread,team_spread_odds,team_moneyline,opponent_team,is_home,is_favorite


In [50]:
unclassified

[]